# Characteristic Function Families on SPY Data

Compare Gaussian, NIG, CGMY, and Lévy-stable fits on SPY returns. We compute AIC and plot CF distance to highlight model quality differences.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'cfad').exists():
    ROOT = ROOT.parent
if not (ROOT / 'cfad').exists():
    raise RuntimeError('Unable to locate project root for cfad')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cfad.utils import load_spy_sample
from cfad.models.gaussian import GaussianCF
from cfad.models.nig import NIGCF
from cfad.models.cgmy import CGMYCF
from cfad.models.levy_stable import LevyStableCF
from cfad.empirical_cf import ecf_at

## Load SPY returns

We use daily SPY returns from 2018 to 2022 and compare parametric characteristic function fits.

In [2]:
returns = load_spy_sample()
returns = returns.loc['2019-01-01':'2021-12-31']
returns.head()

C:\Users\diogo\work_code\cfad\cfad\utils.py:113: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(


Ticker,SPY
Date,
2019-01-02,0.001040
2019-01-03,-0.023863
2019-01-04,0.033496
2019-01-07,0.007885
2019-01-08,0.009395


## Fit models and compare AIC

In [3]:
xi = np.linspace(-15, 15, 256)
ecf = ecf_at(returns.values, xi)
models = [
    GaussianCF(),
    NIGCF(),
    CGMYCF(),
    LevyStableCF(),
]
results = []
for model in models:
    fitted = model.fit(returns.values)
    dist = np.mean(np.abs(ecf - fitted.cf(xi))**2)
    results.append({
        'model': type(fitted).__name__,
        'aic': fitted.aic(returns.values),
        'ecf_l2': dist,
        'repr': repr(fitted),
    })
df = pd.DataFrame(results)
df_sorted = df.sort_values('ecf_l2')
df_sorted

,model,aic,ecf_l2,repr
1,NIGCF,8.000001,6.693394e-12,"NIGCF(alpha=31.7813, beta=-3.8512, delta=0.005..."
0,GaussianCF,4.002201,3.190759e-07,"GaussianCF(mu=0.001011, sigma=0.013799)"
2,CGMYCF,8.008973,2.123072e-06,"CGMYCF(C=0.0000, G=9.3342, M=0.0268, Y=2.0000)"
3,LevyStableCF,8.056638,4.202814e-05,"LevyStableCF(alpha=1.5100, beta=-0.2277, c=0.0..."


## CF distance comparison

Plot the difference between the empirical characteristic function and each fitted model.

In [4]:
fig, ax = plt.subplots(figsize=(10, 6))
for model in models:
    fit_model = model.fit(returns.values)
    ax.plot(xi, np.abs(ecf - fit_model.cf(xi)), label=type(fit_model).__name__)
ax.set_xlabel(r'i')
ax.set_ylabel('ECF L2 error')
ax.set_title('ECF distance for fitted CF families')
ax.legend()
ax.grid(True, alpha=0.3)
figure_path = Path('paper/figures/02_cf_distance.png')
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=150)
print(f'Saved CF distance figure to {figure_path}')

Saved CF distance figure to paper\figures\02_cf_distance.png
